# Deteksi Inkonsistensi Sentimen dan Rating pada Review Aplikasi Mobile Indonesia

Notebook ini mencakup dua bagian utama:
1. **Data Collection** - Scraping review dari Google Play Store
2. **Preprocessing** - Pembersihan, normalisasi, dan labeling teks

## Bagian 1: Data Collection

### 1.1 Instalasi Library

In [1]:
import subprocess
import sys

packages = [
    'google-play-scraper',
    'pandas',
    'numpy',
    'langdetect',
    'tqdm',
    'torch',
    'transformers'
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Semua library berhasil diinstall.')

Installing google-play-scraper...
Installing pandas...
Installing numpy...
Installing langdetect...
Installing tqdm...
Installing torch...
Installing transformers...
Semua library berhasil diinstall.


### 1.2 Import Library

In [2]:
import pandas as pd
import numpy as np
import re
import time
import warnings
import torch
from datetime import datetime

from google_play_scraper import reviews, Sort
from langdetect import detect, LangDetectException
from transformers import pipeline
from tqdm import tqdm

warnings.filterwarnings('ignore')

print('Semua library berhasil diimport.')
print(f'Pandas version     : {pd.__version__}')
print(f'Numpy version      : {np.__version__}')
print(f'PyTorch version    : {torch.__version__}')
print(f'GPU tersedia       : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU                : {torch.cuda.get_device_name(0)}')

Semua library berhasil diimport.
Pandas version     : 3.0.1
Numpy version      : 2.4.6
PyTorch version    : 2.11.0+cu128
GPU tersedia       : True
GPU                : NVIDIA GeForce RTX 5070 Ti Laptop GPU


### 1.3 Konfigurasi Aplikasi Target

In [3]:
APPS = [
    {'name': 'Gojek',      'app_id': 'com.gojek.app'},
    {'name': 'Tokopedia',  'app_id': 'com.tokopedia.tkpd'},
    {'name': 'Shopee',     'app_id': 'com.shopee.id'},
    {'name': 'DANA',       'app_id': 'id.dana'},
    {'name': 'BCA Mobile', 'app_id': 'com.bca'},
]

SCRAPE_COUNT  = 5000
MAX_RETRY     = 3
DELAY_SECONDS = 2
MIN_TOTAL     = 10000
EXTRA_COUNT   = 500

print(f'Total aplikasi target : {len(APPS)}')
print(f'Review per aplikasi   : {SCRAPE_COUNT:,}')
print(f'Target total review   : {SCRAPE_COUNT * len(APPS):,} (sebelum filtering)')

Total aplikasi target : 5
Review per aplikasi   : 5,000
Target total review   : 25,000 (sebelum filtering)


### 1.4 Fungsi Scraping dengan Retry Logic

In [4]:
def scrape_app_reviews(app_name, app_id, count=5000, max_retry=3):
    for attempt in range(1, max_retry + 1):
        try:
            print(f'  Scraping {app_name} ({app_id}) - percobaan {attempt}/{max_retry}...')
            result, _ = reviews(
                app_id,
                lang='id',
                country='id',
                sort=Sort.NEWEST,
                count=count
            )
            records = []
            for r in result:
                records.append({
                    'app':       app_name,
                    'app_id':    app_id,
                    'username':  r.get('userName', ''),
                    'rating':    r.get('score', np.nan),
                    'text':      r.get('content', ''),
                    'thumbs_up': r.get('thumbsUpCount', 0),
                    'date':      r.get('at', None),
                    'reply':     r.get('replyContent', ''),
                })
            print(f'  Berhasil mengambil {len(records):,} review dari {app_name}.')
            return records
        except Exception as e:
            print(f'  Gagal (percobaan {attempt}): {e}')
            if attempt < max_retry:
                print(f'  Menunggu {DELAY_SECONDS} detik sebelum retry...')
                time.sleep(DELAY_SECONDS)
    print(f'  PERINGATAN: Gagal scraping {app_name} setelah {max_retry} percobaan.')
    return []

print('Fungsi scraping siap.')

Fungsi scraping siap.


### 1.5 Proses Scraping Utama

In [5]:
all_records = []
app_record_counts = {}

print('=' * 60)
print('MEMULAI PROSES SCRAPING')
print('=' * 60)

for app in tqdm(APPS, desc='Progress scraping'):
    print(f'\nMulai scraping: {app["name"]}')
    records = scrape_app_reviews(
        app_name=app['name'],
        app_id=app['app_id'],
        count=SCRAPE_COUNT,
        max_retry=MAX_RETRY
    )
    all_records.extend(records)
    app_record_counts[app['name']] = len(records)
    print(f'Total review terkumpul sejauh ini: {len(all_records):,}')
    time.sleep(DELAY_SECONDS)

print('\n' + '=' * 60)
print(f'SCRAPING SELESAI. Total raw review: {len(all_records):,}')
print('=' * 60)
for app_name, count in app_record_counts.items():
    print(f'  {app_name:<15}: {count:,} review')

MEMULAI PROSES SCRAPING


Progress scraping:   0%|          | 0/5 [00:00<?, ?it/s]


Mulai scraping: Gojek
  Scraping Gojek (com.gojek.app) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari Gojek.
Total review terkumpul sejauh ini: 5,000


Progress scraping:  20%|██        | 1/5 [00:03<00:15,  3.96s/it]


Mulai scraping: Tokopedia
  Scraping Tokopedia (com.tokopedia.tkpd) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari Tokopedia.
Total review terkumpul sejauh ini: 10,000


...


Mulai scraping: DANA
  Scraping DANA (id.dana) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari DANA.
Total review terkumpul sejauh ini: 20,000


Progress scraping:  80%|████████  | 4/5 [00:15<00:03,  3.84s/it]


Mulai scraping: BCA Mobile
  Scraping BCA Mobile (com.bca) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari BCA Mobile.
Total review terkumpul sejauh ini: 25,000


Progress scraping: 100%|██████████| 5/5 [00:19<00:00,  3.86s/it]


SCRAPING SELESAI. Total raw review: 25,000
  Gojek          : 5,000 review
  Tokopedia      : 5,000 review
  Shopee         : 5,000 review
  DANA           : 5,000 review
  BCA Mobile     : 5,000 review


### 1.6 Simpan Raw Data

In [6]:
df_raw = pd.DataFrame(all_records)

print(f'Shape raw DataFrame  : {df_raw.shape}')
print(f'Kolom yang tersedia  : {list(df_raw.columns)}')
print('\nContoh data (5 baris pertama):')
display(df_raw.head())

df_raw.to_csv('raw_reviews.csv', index=False, encoding='utf-8-sig')
print('\nRaw data berhasil disimpan ke raw_reviews.csv')

Shape raw DataFrame  : (25000, 8)
Kolom yang tersedia  : ['app', 'app_id', 'username', 'rating', 'text', 'thumbs_up', 'date', 'reply']

Contoh data (5 baris pertama):


,app,app_id,username,rating,text,thumbs_up,date,reply
0,Gojek,com.gojek.app,Pengguna Google,5,lebih cepat jemput,0,2026-05-31 06:41:56,NaN
1,Gojek,com.gojek.app,Pengguna Google,5,sangat oke... baik diperjalanan sangat membant...,0,2026-05-31 06:07:45,NaN
2,Gojek,com.gojek.app,Pengguna Google,5,oke,0,2026-05-31 06:00:02,NaN
3,Gojek,com.gojek.app,Pengguna Google,5,cepat dan ga ribet,0,2026-05-31 05:53:42,NaN
4,Gojek,com.gojek.app,Pengguna Google,5,ok,0,2026-05-31 05:28:42,NaN



Raw data berhasil disimpan ke raw_reviews.csv


## Bagian 2: Preprocessing

### 2.1 Load Data dan Inisialisasi

In [7]:
df = df_raw.copy()

print(f'Data dimuat: {len(df):,} baris, {df.shape[1]} kolom')
print('\nInfo kolom:')
print(df.dtypes)
print('\nJumlah missing values per kolom:')
print(df.isnull().sum())

Data dimuat: 25,000 baris, 8 kolom

Info kolom:
app                     str
app_id                  str
username                str
rating                int64
text                    str
thumbs_up             int64
date         datetime64[us]
reply                   str
dtype: object

Jumlah missing values per kolom:
app              0
app_id           0
username         0
rating           0
text             0
thumbs_up        0
date             0
reply        12589
dtype: int64


### 2.2 Filter Review Kosong dan Terlalu Pendek

In [8]:
before = len(df)

df = df[df['text'].notna()]
df = df[df['text'].str.strip() != '']

df['_wc_temp'] = df['text'].apply(lambda x: len(str(x).split()))
df = df[df['_wc_temp'] >= 5].drop(columns=['_wc_temp'])
df = df.reset_index(drop=True)

after = len(df)
print(f'Filter teks kosong/pendek:')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} review')

Filter teks kosong/pendek:
  Sebelum : 25,000
  Sesudah : 13,720
  Dihapus : 11,280 review


### 2.3 Filter Bahasa Indonesia

In [9]:
before = len(df)

def detect_language(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return 'unknown'
    except Exception:
        return 'unknown'

tqdm.pandas(desc='Deteksi bahasa')
df['lang'] = df['text'].progress_apply(detect_language)

print('\nDistribusi bahasa yang terdeteksi:')
print(df['lang'].value_counts().head(10))

df = df[df['lang'] == 'id'].reset_index(drop=True)

after = len(df)
print(f'\nFilter bahasa Indonesia:')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} review')

Deteksi bahasa: 100%|██████████| 13720/13720 [00:17<00:00, 784.45it/s]


Distribusi bahasa yang terdeteksi:
lang
id    12856
tl      339
de      186
en      104
so       31
hr       27
et       27
no       19
ca       15
sl       14
Name: count, dtype: int64

Filter bahasa Indonesia:
  Sebelum : 13,720
  Sesudah : 12,856
  Dihapus : 864 review


### 2.4 Normalisasi Teks

In [10]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F926-\U0001F937"
    "\U00010000-\U0010FFFF"
    "]+",
    flags=re.UNICODE
)

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = EMOJI_PATTERN.sub('', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_original'] = df['text']

tqdm.pandas(desc='Normalisasi teks')
df['text'] = df['text'].progress_apply(normalize_text)

print('Normalisasi teks selesai.')
print('\nContoh sebelum dan sesudah normalisasi (3 sampel):')
for i in df.sample(3, random_state=42).index:
    print(f'  Asli  : {df.loc[i, "text_original"][:80]}')
    print(f'  Norma : {df.loc[i, "text"][:80]}')
    print()

Normalisasi teks: 100%|██████████| 12856/12856 [00:00<00:00, 143192.81it/s]

Normalisasi teks selesai.

Contoh sebelum dan sesudah normalisasi (3 sampel):
  Asli  : dana instan tidak bisa di pakai semenjak mitra nya pindah ke ADAKAMI padahal say
  Norma : dana instan tidak bisa di pakai semenjak mitra nya pindah ke adakami padahal say

  Asli  : Tutup aja omon omon susah pecairan driver bayar sendiri payah
  Norma : tutup aja omon omon susah pecairan driver bayar sendiri payah

  Asli  : mapsnya kadang ga jalan, ga sesuai. kyk tiba² ilang mapsnya/ke cencel padahal te
  Norma : mapsnya kadang ga jalan, ga sesuai. kyk tiba² ilang mapsnya/ke cencel padahal te



### 2.5 Hapus Duplikat

In [11]:
before = len(df)
df = df.drop_duplicates(subset=['text', 'app'], keep='first').reset_index(drop=True)
after = len(df)
print(f'Hapus duplikat (berdasarkan text + app):')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} duplikat')

Hapus duplikat (berdasarkan text + app):
  Sebelum : 12,856
  Sesudah : 12,846
  Dihapus : 10 duplikat


### 2.6 Tambah Kolom word_count

In [12]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

print('Kolom word_count berhasil ditambahkan.')
print(f'\nStatistik word_count:')
print(df['word_count'].describe().round(2))

Kolom word_count berhasil ditambahkan.

Statistik word_count:
count    12846.00
mean        19.02
std         15.85
min          4.00
25%          8.00
50%         14.00
75%         24.00
max         97.00
Name: word_count, dtype: float64


### 2.7 Validasi Total Review dan Scraping Tambahan

In [13]:
iteration = 0

while len(df) < MIN_TOTAL:
    iteration += 1
    print('=' * 60)
    print(f'PERINGATAN: Total review = {len(df):,} < minimum {MIN_TOTAL:,}')
    print(f'Memulai iterasi tambahan ke-{iteration}...')
    print('=' * 60)

    apps_sorted = df['app'].value_counts().sort_values().head(2).index.tolist()
    print(f'Aplikasi yang akan di-scrape tambahan: {apps_sorted}')

    new_records = []
    for app_name in apps_sorted:
        app_cfg = next((a for a in APPS if a['name'] == app_name), None)
        if app_cfg is None:
            continue
        extra = scrape_app_reviews(app_cfg['name'], app_cfg['app_id'], EXTRA_COUNT, MAX_RETRY)
        new_records.extend(extra)
        time.sleep(DELAY_SECONDS)

    if not new_records:
        print('Tidak ada review tambahan. Menghentikan loop.')
        break

    df_extra = pd.DataFrame(new_records)
    df_extra = df_extra[df_extra['text'].notna()]
    df_extra = df_extra[df_extra['text'].str.strip() != '']
    df_extra['_wc'] = df_extra['text'].apply(lambda x: len(str(x).split()))
    df_extra = df_extra[df_extra['_wc'] >= 5].drop(columns=['_wc'])

    tqdm.pandas(desc='Deteksi bahasa (tambahan)')
    df_extra['lang'] = df_extra['text'].progress_apply(detect_language)
    df_extra = df_extra[df_extra['lang'] == 'id']

    df_extra['text_original'] = df_extra['text']
    tqdm.pandas(desc='Normalisasi teks (tambahan)')
    df_extra['text'] = df_extra['text'].progress_apply(normalize_text)
    df_extra['word_count'] = df_extra['text'].apply(lambda x: len(str(x).split()))

    df = pd.concat([df, df_extra], ignore_index=True)
    df = df.drop_duplicates(subset=['text', 'app'], keep='first').reset_index(drop=True)
    print(f'Total review setelah iterasi {iteration}: {len(df):,}')

print('\n' + '=' * 60)
if len(df) >= MIN_TOTAL:
    print(f'Validasi LULUS: Total review = {len(df):,}')
else:
    print(f'Validasi GAGAL: Total review = {len(df):,}')
print('=' * 60)


Validasi LULUS: Total review = 12,846


### 2.8 Labeling dengan Pretrained Sentiment Model

Sentimen teks dideteksi menggunakan model `mdhugol/indonesia-bert-sentiment-classification` (IndoBERT fine-tuned untuk sentimen Bahasa Indonesia). Label inkonsistensi diturunkan dengan membandingkan sentimen teks hasil model dengan sentimen bintang dari rating pengguna.

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `text_sentiment` | string | POSITIVE / NEGATIVE / NEUTRAL |
| `sentiment_confidence` | float | Confidence score model (0-1) |
| `label` | int | 0 = konsisten, 1 = inkonsisten |

In [14]:
SENTIMENT_MODEL = 'mdhugol/indonesia-bert-sentiment-classification'
LABEL_MAP = {'LABEL_0': 'POSITIVE', 'LABEL_1': 'NEGATIVE', 'LABEL_2': 'NEUTRAL'}
INFER_BATCH = 64

device = 0 if torch.cuda.is_available() else -1
print(f'Device inferensi : {"GPU (cuda:0)" if device == 0 else "CPU"}')
print(f'Memuat model     : {SENTIMENT_MODEL}...')

sentiment_pipe = pipeline(
    'text-classification',
    model=SENTIMENT_MODEL,
    device=device,
    batch_size=INFER_BATCH
)

print('Model berhasil dimuat.')

Device inferensi : GPU (cuda:0)
Memuat model     : mdhugol/indonesia-bert-sentiment-classification...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model berhasil dimuat.


In [15]:
texts = df['text'].tolist()
raw_outputs = []

print(f'Mulai inferensi sentimen untuk {len(texts):,} review (batch size={INFER_BATCH})...')

for i in tqdm(range(0, len(texts), INFER_BATCH), desc='Inferensi sentimen'):
    batch = texts[i : i + INFER_BATCH]
    outputs = sentiment_pipe(batch, truncation=True, max_length=512)
    raw_outputs.extend(outputs)

df['text_sentiment']      = [LABEL_MAP.get(o['label'], 'NEUTRAL') for o in raw_outputs]
df['sentiment_confidence']= [round(o['score'], 4) for o in raw_outputs]

print('\nDistribusi text_sentiment:')
print(df['text_sentiment'].value_counts())
print(f'\nRata-rata confidence score: {df["sentiment_confidence"].mean():.4f}')

Mulai inferensi sentimen untuk 12,846 review (batch size=64)...


Inferensi sentimen: 100%|██████████| 201/201 [00:21<00:00,  9.56it/s]


Distribusi text_sentiment:
text_sentiment
NEUTRAL     7870
POSITIVE    3152
NEGATIVE    1824
Name: count, dtype: int64

Rata-rata confidence score: 0.9305


In [16]:
def star_to_sentiment(rating):
    if rating >= 4:
        return 'POSITIVE'
    if rating <= 2:
        return 'NEGATIVE'
    return 'NEUTRAL'

df['_star_sentiment'] = df['rating'].apply(star_to_sentiment)

def compute_label(row):
    star = row['_star_sentiment']
    text = row['text_sentiment']
    # Jika salah satu NEUTRAL, tidak bisa dianggap inkonsisten
    if star == 'NEUTRAL' or text == 'NEUTRAL':
        return 0
    return 1 if star != text else 0

df['label'] = df.apply(compute_label, axis=1)
df = df.drop(columns=['_star_sentiment'])

n_incon = df['label'].sum()
n_total = len(df)

print('Distribusi label final:')
print(f'  Konsisten   (0): {n_total - n_incon:,} ({(1 - n_incon/n_total)*100:.2f}%)')
print(f'  Inkonsisten (1): {n_incon:,} ({n_incon/n_total*100:.2f}%)')

print('\nCross-tabulation rating vs text_sentiment:')
display(
    df.groupby(['rating', 'text_sentiment'])['label']
      .count()
      .unstack(fill_value=0)
)

Distribusi label final:
  Konsisten   (0): 11,802 (91.87%)
  Inkonsisten (1): 1,044 (8.13%)

Cross-tabulation rating vs text_sentiment:


text_sentiment,NEGATIVE,NEUTRAL,POSITIVE
rating,,,
1,589,5151,144
2,166,855,29
3,198,725,37
4,204,388,135
5,667,751,2807


### 2.9 Simpan Hasil Preprocessing

In [17]:
cols_export = [
    'app', 'app_id', 'username', 'rating', 'text', 'text_original',
    'thumbs_up', 'date', 'reply', 'lang', 'word_count',
    'text_sentiment', 'sentiment_confidence', 'label'
]
cols_export = [c for c in cols_export if c in df.columns]

df_clean = df[cols_export].copy()
df_clean.to_csv('clean_reviews.csv', index=False, encoding='utf-8-sig')

print(f'Data bersih berhasil disimpan ke clean_reviews.csv')
print(f'Shape final DataFrame : {df_clean.shape}')
print(f'Kolom yang disimpan   : {list(df_clean.columns)}')
print('\nContoh data final (3 baris):')
display(df_clean.head(3))

Data bersih berhasil disimpan ke clean_reviews.csv
Shape final DataFrame : (12846, 14)
Kolom yang disimpan   : ['app', 'app_id', 'username', 'rating', 'text', 'text_original', 'thumbs_up', 'date', 'reply', 'lang', 'word_count', 'text_sentiment', 'sentiment_confidence', 'label']

Contoh data final (3 baris):


,app,app_id,username,rating,text,text_original,thumbs_up,date,reply,lang,word_count,text_sentiment,sentiment_confidence,label
0,Gojek,com.gojek.app,Pengguna Google,5,sangat oke.. baik diperjalanan sangat membantu...,sangat oke... baik diperjalanan sangat membant...,0,2026-05-31 06:07:45,NaN,id,8,POSITIVE,0.9948,0
1,Gojek,com.gojek.app,Pengguna Google,5,baik & bagus servis nya,baik & bagus servis nya 👍👍,0,2026-05-31 05:22:59,NaN,id,5,POSITIVE,0.9951,0
2,Gojek,com.gojek.app,Pengguna Google,2,tiap hari langganan go-jek ke mana-mana tapi n...,tiap hari langganan Go-Jek ke mana-mana tapi n...,0,2026-05-31 05:05:10,"Mohon maaf atas kendalanya, Kak Ragil. Kesulit...",id,19,NEUTRAL,0.9886,0


## Bagian 3: Eksplorasi Data

### 3.1 Distribusi Rating per Aplikasi

In [18]:
print('DISTRIBUSI RATING PER APLIKASI')
print('=' * 60)

rating_dist = df_clean.groupby(['app', 'rating']).size().unstack(fill_value=0)
rating_dist.columns = [f'Rating {c}' for c in rating_dist.columns]
rating_dist['Total'] = rating_dist.sum(axis=1)
display(rating_dist)

print('\nRata-rata rating per aplikasi:')
print(df_clean.groupby('app')['rating'].mean().round(2).to_string())

DISTRIBUSI RATING PER APLIKASI


,Rating 1,Rating 2,Rating 3,Rating 4,Rating 5,Total
app,,,,,,
BCA Mobile,1653,358,304,175,599,3089
DANA,754,158,152,143,865,2072
Gojek,884,164,142,138,805,2133
Shopee,789,136,135,130,1338,2528
Tokopedia,1804,234,227,141,618,3024



Rata-rata rating per aplikasi:
app
BCA Mobile    2.26
DANA          3.10
Gojek         2.91
Shopee        3.43
Tokopedia     2.18


### 3.2 Distribusi Label per Aplikasi

In [19]:
print('DISTRIBUSI LABEL (INKONSISTEN vs KONSISTEN) PER APLIKASI')
print('=' * 60)

label_tbl = df_clean.groupby('app')['label'].agg(
    Total='count',
    Inkonsisten='sum'
).assign(Konsisten=lambda x: x['Total'] - x['Inkonsisten'])
label_tbl['Pct_Inkonsisten (%)'] = (label_tbl['Inkonsisten'] / label_tbl['Total'] * 100).round(2)
display(label_tbl)

print('\nDistribusi text_sentiment per aplikasi:')
display(
    df_clean.groupby(['app', 'text_sentiment']).size()
            .unstack(fill_value=0)
)

DISTRIBUSI LABEL (INKONSISTEN vs KONSISTEN) PER APLIKASI


,Total,Inkonsisten,Konsisten,Pct_Inkonsisten (%)
app,,,,
BCA Mobile,3089,239,2850,7.74
DANA,2072,283,1789,13.66
Gojek,2133,160,1973,7.50
Shopee,2528,154,2374,6.09
Tokopedia,3024,208,2816,6.88



Distribusi text_sentiment per aplikasi:


text_sentiment,NEGATIVE,NEUTRAL,POSITIVE
app,,,
BCA Mobile,556,2208,325
DANA,477,1149,446
Gojek,221,1193,719
Shopee,229,1156,1143
Tokopedia,341,2164,519


### 3.3 Rata-rata Word Count per Segmen Teks

In [20]:
def text_segment(wc):
    if wc < 20:
        return 'Pendek (< 20 kata)'
    elif wc <= 50:
        return 'Sedang (20-50 kata)'
    else:
        return 'Panjang (> 50 kata)'

df_clean['text_segment'] = df_clean['word_count'].apply(text_segment)

print('RATA-RATA WORD COUNT PER SEGMEN TEKS')
print('=' * 60)

segment_stats = df_clean.groupby('text_segment').agg(
    Jumlah_Review=('word_count', 'count'),
    Rata_rata=('word_count', 'mean'),
    Min=('word_count', 'min'),
    Max=('word_count', 'max')
).round(2)

seg_order = ['Pendek (< 20 kata)', 'Sedang (20-50 kata)', 'Panjang (> 50 kata)']
display(segment_stats.reindex([s for s in seg_order if s in segment_stats.index]))

print('\nDistribusi segmen:')
seg_dist = df_clean['text_segment'].value_counts()
for seg in seg_order:
    if seg in seg_dist:
        print(f'  {seg:<25}: {seg_dist[seg]:,} ({seg_dist[seg]/len(df_clean)*100:.1f}%)')

RATA-RATA WORD COUNT PER SEGMEN TEKS


,Jumlah_Review,Rata_rata,Min,Max
text_segment,,,,
Pendek (< 20 kata),8514,10.32,4,19
Sedang (20-50 kata),3589,29.77,20,50
Panjang (> 50 kata),743,66.76,51,97



Distribusi segmen:
  Pendek (< 20 kata)       : 8,514 (66.3%)
  Sedang (20-50 kata)      : 3,589 (27.9%)
  Panjang (> 50 kata)      : 743 (5.8%)


### 3.4 Ringkasan Akhir Dataset

In [21]:
print('RINGKASAN AKHIR DATASET')
print('=' * 60)
print(f'Total review final           : {len(df_clean):,}')
print(f'Jumlah aplikasi              : {df_clean["app"].nunique()}')
print(f'Rentang rating               : {df_clean["rating"].min()} - {df_clean["rating"].max()}')
print(f'Rata-rata word count         : {df_clean["word_count"].mean():.2f} kata')
print(f'Total review inkonsisten     : {df_clean["label"].sum():,} ({df_clean["label"].mean()*100:.2f}%)')
print()
print('Distribusi per aplikasi:')
print(df_clean['app'].value_counts().to_string())
print()
print('File yang dihasilkan:')
print('  - raw_reviews.csv   : data mentah dari scraping')
print('  - clean_reviews.csv : data bersih + label, siap untuk pemodelan')

RINGKASAN AKHIR DATASET
Total review final           : 12,846
Jumlah aplikasi              : 5
Rentang rating               : 1 - 5
Rata-rata word count         : 19.02 kata
Total review inkonsisten     : 1,044 (8.13%)

Distribusi per aplikasi:
app
BCA Mobile    3089
Tokopedia     3024
Shopee        2528
Gojek         2133
DANA          2072

File yang dihasilkan:
  - raw_reviews.csv   : data mentah dari scraping
  - clean_reviews.csv : data bersih + label, siap untuk pemodelan
